# Requête HTTP 

Un requête HTTP est une requête basée sur le protocole TCP, elle fait partie de la couche application de la couche OSI. Elle permet d'accéder aux données mise à disposition sur une adresse IP (ou url résolue par un DNS) et un port. 

Les deux ports les plus utilisés dans le web sont le 80 pour les sites en HTTP et le 443 pour les sites en HTTPS. HTTPS est une variable du protocole HTTP basé sur le protocole TLS.

Il existe de nombreux types de requêtes selon la convention `REST`: 
- GET
- POST
- PUT 
- DELETE
- UPDATE.

Dans notre cas, nous allons utiliser la plupart du temps des GET et potentiellement des POST. 
- Le GET permet comme son nom l'indique de récupérer des informations en fonction de certains paramètres. 
- Le POST nécessite un envoi de données pour récupérer des données. Le body du post est, la plupart du temps, envoyé sous la forme d'un objet JSON.

Ces requêtes encapsulent un certain nombre de paramètres qui permettent soient d'identifier une provenance et un utilisateur ou de réaliser différentes actions.

In [3]:
import requests

In [4]:
url = "https://www.esiee.fr/"
response = requests.get(url)
response.status_code

200

Il existe deux méthodes pour récupérer le contenu de la page :

- `response.text` qui permet de retourner le texte sous la forme d'une chaine de charactères.
- `response.content` qui permet de récupérer le contenu de la page sous la forme de bytes

In [5]:
type(response.content)

bytes

In [6]:
type(response.text)

str

Pour récupérer les 1000 premiers charactères de la page :

In [7]:
response.text[0:1000]

'<!DOCTYPE html>\n<html lang="fr-FR">\n<head>\n\n<meta charset="utf-8">\n<!-- \n\tThis website is powered by TYPO3 - inspiring people to share!\n\tTYPO3 is a free open source Content Management Framework initially created by Kasper Skaarhoj and licensed under GNU/GPL.\n\tTYPO3 is copyright 1998-2025 of Kasper Skaarhoj. Extensions are copyright of their respective owners.\n\tInformation and contribution at https://typo3.org/\n-->\n\n\n\n<title>ESIEE Paris, l&#039;école de l&#039;innovation technologique | ESIEE Paris</title>\n<meta name="generator" content="TYPO3 CMS" />\n<meta name="description" content="Rejoignez ESIEE Paris, grande école d&#039;ingénieur dans les domaines des transitions numérique, énergétique et environnementale. Classée dans le groupe A, parmi les meilleures écoles d&#039;ingénieur selon le classement de l&#039;Etudiant. Habilitée par la Commission des Titres d&#039;Ingénieur (CTI). Membre de la Conférence des Grandes Ecoles (CGE). " />\n<meta name="viewport" conte

Pour récupérer les headers HTTP de la réponse :

In [8]:
response.headers

{'Date': 'Tue, 18 Nov 2025 14:53:33 GMT', 'Server': 'Apache', 'Content-Language': 'fr', 'Vary': 'Accept-Encoding', 'Content-Encoding': 'gzip', 'X-UA-Compatible': 'IE=edge', 'X-Content-Type-Options': 'nosniff', 'Content-Length': '16642', 'Content-Type': 'text/html; charset=utf-8', 'X-Varnish': '536447799 536708222', 'Age': '111', 'Via': '1.1 varnish (Varnish/7.1)', 'Accept-Ranges': 'bytes', 'Connection': 'keep-alive'}

On peut modifier les paramètres de la requête et/ou ses headers. On peut par exemple ajouter un UserAgent (identifiant de l'initiateur de la requête) et un timeout de 10 secondes :

In [9]:
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
response = requests.get(url, headers=headers, timeout = 10)
response.content[0:1000]

b'<!DOCTYPE html>\n<html lang="fr-FR">\n<head>\n\n<meta charset="utf-8">\n<!-- \n\tThis website is powered by TYPO3 - inspiring people to share!\n\tTYPO3 is a free open source Content Management Framework initially created by Kasper Skaarhoj and licensed under GNU/GPL.\n\tTYPO3 is copyright 1998-2025 of Kasper Skaarhoj. Extensions are copyright of their respective owners.\n\tInformation and contribution at https://typo3.org/\n-->\n\n\n\n<title>ESIEE Paris, l&#039;\xc3\xa9cole de l&#039;innovation technologique | ESIEE Paris</title>\n<meta name="generator" content="TYPO3 CMS" />\n<meta name="description" content="Rejoignez ESIEE Paris, grande \xc3\xa9cole d&#039;ing\xc3\xa9nieur dans les domaines des transitions num\xc3\xa9rique, \xc3\xa9nerg\xc3\xa9tique et environnementale. Class\xc3\xa9e dans le groupe A, parmi les meilleures \xc3\xa9coles d&#039;ing\xc3\xa9nieur selon le classement de l&#039;Etudiant. Habilit\xc3\xa9e par la Commission des Titres d&#039;Ing\xc3\xa9nieur (CTI). Membr

## Exercice

## Exercice 1

- Créer une classe Python permettant de faire des requêtes HTTP.
- Cette classe doit utiliser toujours le même UserAgent.
- Le TimeOut sera spécifié à chaque appelle avec une valeur par défaut.
- Un mécanisme de retry sera mis en place de façon recursive.

## Exercice 2

- Faire une fonction permettant de supprimer tous les espaces supperflus d'une string
- Faire une fonction qui prend une string html et renvois une string intelligible (enlever les caractères spéciaux,
- Récupérer le domaine en fonction d'un url

In [10]:
## Exercice 1 : Classe HTTPClient avec retry récursif

import requests
import time

class HTTPClient:
    """
    Classe pour effectuer des requêtes HTTP avec retry automatique
    """
    def __init__(self, user_agent=None):
        # UserAgent par défaut utilisé pour toutes les requêtes
        self.user_agent = user_agent or 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    
    def get(self, url, timeout=10, max_retries=3, retry_count=0):
        """
        Effectue une requête GET avec mécanisme de retry récursif
        
        Args:
            url: URL à requêter
            timeout: Temps maximum d'attente (par défaut 10s)
            max_retries: Nombre maximum de tentatives
            retry_count: Compteur de tentatives (utilisé par la récursion)
        
        Returns:
            Response object ou None en cas d'échec
        """
        headers = {'User-Agent': self.user_agent}
        
        try:
            print(f"Tentative {retry_count + 1}/{max_retries + 1} pour {url}")
            response = requests.get(url, headers=headers, timeout=timeout)
            response.raise_for_status()  # Lève une exception si code HTTP d'erreur
            print(f"✓ Succès! Status code: {response.status_code}")
            return response
            
        except requests.exceptions.RequestException as e:
            print(f"✗ Erreur: {type(e).__name__}")
            
            # Si on n'a pas dépassé le nombre max de retries, on réessaye récursivement
            if retry_count < max_retries:
                wait_time = 2 ** retry_count  # Backoff exponentiel: 1s, 2s, 4s...
                print(f"  → Nouvelle tentative dans {wait_time}s...")
                time.sleep(wait_time)
                # Appel récursif avec retry_count incrémenté
                return self.get(url, timeout, max_retries, retry_count + 1)
            else:
                print(f"✗ Échec définitif après {max_retries + 1} tentatives")
                return None

# ===== TESTS DE LA CLASSE =====

print("=" * 70)
print("TEST 1: Requête réussie sur URL valide")
print("=" * 70)
client = HTTPClient()
response = client.get("https://www.esiee.fr/", timeout=10)
if response:
    print(f"✓ Contenu récupéré: {len(response.content)} bytes")
    print(f"✓ Content-Type: {response.headers.get('Content-Type', 'N/A')}")

print("\n" + "=" * 70)
print("TEST 2: Retry automatique (timeout court, puis normal)")
print("=" * 70)
# Ce test montre le retry en action avec un timeout réaliste
response = client.get("https://httpbin.org/delay/1", timeout=0.5, max_retries=2)
if response:
    print(f"✓ Requête réussie après retry")

print("\n" + "=" * 70)
print("TEST 3: Échec après plusieurs retries")
print("=" * 70)
# URL qui n'existe pas pour tester l'échec complet
response = client.get("https://site-inexistant-test-12345.invalide/", timeout=3, max_retries=2)
if not response:
    print("✓ Gestion correcte de l'échec après retries")

print("\n" + "=" * 70)
print("TEST 4: Utilisation avec un User-Agent personnalisé")
print("=" * 70)
custom_client = HTTPClient(user_agent="MonBot/1.0")
response = custom_client.get("https://httpbin.org/user-agent", timeout=10)
if response:
    print(f"✓ User-Agent vérifié: {response.json()}")

TEST 1: Requête réussie sur URL valide
Tentative 1/4 pour https://www.esiee.fr/
✓ Succès! Status code: 200
✓ Contenu récupéré: 92791 bytes
✓ Content-Type: text/html; charset=utf-8

TEST 2: Retry automatique (timeout court, puis normal)
Tentative 1/3 pour https://httpbin.org/delay/1
✓ Succès! Status code: 200
✓ Contenu récupéré: 92791 bytes
✓ Content-Type: text/html; charset=utf-8

TEST 2: Retry automatique (timeout court, puis normal)
Tentative 1/3 pour https://httpbin.org/delay/1
✗ Erreur: ReadTimeout
  → Nouvelle tentative dans 1s...
✗ Erreur: ReadTimeout
  → Nouvelle tentative dans 1s...
Tentative 2/3 pour https://httpbin.org/delay/1
Tentative 2/3 pour https://httpbin.org/delay/1
✗ Erreur: ReadTimeout
  → Nouvelle tentative dans 2s...
✗ Erreur: ReadTimeout
  → Nouvelle tentative dans 2s...
Tentative 3/3 pour https://httpbin.org/delay/1
Tentative 3/3 pour https://httpbin.org/delay/1
✗ Erreur: ReadTimeout
✗ Échec définitif après 3 tentatives

TEST 3: Échec après plusieurs retries
Tent

In [11]:
## Exercice 2 : Fonctions de nettoyage de texte et extraction de domaine

import re
from urllib.parse import urlparse
from bs4 import BeautifulSoup

# Fonction 1: Supprimer les espaces superflus
def remove_extra_spaces(text):
    """
    Supprime tous les espaces superflus d'une string
    - Plusieurs espaces consécutifs → 1 seul espace
    - Espaces en début/fin de ligne
    - Lignes vides multiples
    
    Args:
        text: La chaîne à nettoyer
    
    Returns:
        La chaîne nettoyée
    """
    if not text:
        return ""
    
    # Remplacer les espaces multiples par un seul espace
    text = re.sub(r' +', ' ', text)
    
    # Remplacer les tabulations et sauts de ligne multiples
    text = re.sub(r'\t+', ' ', text)
    text = re.sub(r'\n\s*\n+', '\n\n', text)
    
    # Supprimer les espaces en début et fin de chaque ligne
    lines = [line.strip() for line in text.split('\n')]
    
    # Recombiner et supprimer espaces début/fin
    return '\n'.join(lines).strip()


# Fonction 2: Nettoyer le HTML pour obtenir du texte intelligible
def clean_html_to_text(html_string):
    """
    Convertit une string HTML en texte intelligible
    - Enlève toutes les balises HTML
    - Supprime les caractères spéciaux et entités HTML
    - Nettoie les espaces superflus
    
    Args:
        html_string: La chaîne HTML à nettoyer
    
    Returns:
        Texte propre et lisible
    """
    if not html_string:
        return ""
    
    # Parser le HTML avec BeautifulSoup
    soup = BeautifulSoup(html_string, 'html.parser')
    
    # Supprimer les scripts et styles
    for script in soup(['script', 'style', 'meta', 'link']):
        script.decompose()
    
    # Récupérer le texte
    text = soup.get_text()
    
    # Nettoyer les espaces superflus
    text = remove_extra_spaces(text)
    
    # Supprimer les caractères spéciaux problématiques mais garder la ponctuation
    text = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f\x7f-\x9f]', '', text)
    
    return text


# Fonction 3: Extraire le domaine d'une URL
def extract_domain(url):
    """
    Extrait le nom de domaine d'une URL complète
    
    Args:
        url: L'URL complète (ex: https://www.exemple.com/path?query=1)
    
    Returns:
        Le domaine (ex: www.exemple.com)
    """
    if not url:
        return ""
    
    try:
        # Parser l'URL
        parsed = urlparse(url)
        # Retourner le netloc (network location = domaine)
        domain = parsed.netloc
        
        # Si pas de domaine (URL relative), essayer de parser différemment
        if not domain:
            # Extraire avec regex comme fallback
            match = re.search(r'(?:https?://)?(?:www\.)?([^/]+)', url)
            domain = match.group(1) if match else url
        
        return domain
    except Exception as e:
        print(f"Erreur lors de l'extraction du domaine: {e}")
        return ""


# ===== TESTS DES FONCTIONS =====

print("=" * 60)
print("TEST 1: Suppression des espaces superflus")
print("=" * 60)

test_text = """Ceci    est   un    texte     avec
    
    
beaucoup      d'espaces    superflus  
    et    des   lignes   vides"""

print("Avant:")
print(repr(test_text))
print("\nAprès:")
cleaned = remove_extra_spaces(test_text)
print(repr(cleaned))
print(cleaned)

print("\n" + "=" * 60)
print("TEST 2: Nettoyage HTML vers texte")
print("=" * 60)

html_test = """
<html>
    <head>
        <title>Test Page</title>
        <script>alert('test');</script>
        <style>.test { color: red; }</style>
    </head>
    <body>
        <h1>Titre Principal</h1>
        <p>Ceci est un     paragraphe avec    des espaces.</p>
        <div>
            <span>Texte dans un span</span>
            <a href="#">Un lien</a>
        </div>
        &nbsp;&nbsp;Entités HTML&nbsp;
    </body>
</html>
"""

print("HTML original:")
print(html_test[:200] + "...")
print("\nTexte nettoyé:")
clean_text = clean_html_to_text(html_test)
print(clean_text)

print("\n" + "=" * 60)
print("TEST 3: Extraction de domaine")
print("=" * 60)

test_urls = [
    "https://www.esiee.fr/",
    "http://example.com/path/to/page?query=value&other=123",
    "https://subdomain.example.co.uk/page",
    "www.google.com",
    "https://github.com/user/repo/blob/main/file.py"
]

for url in test_urls:
    domain = extract_domain(url)
    print(f"URL: {url}")
    print(f"→ Domaine: {domain}\n")

TEST 1: Suppression des espaces superflus
Avant:
"Ceci    est   un    texte     avec\n\n\nbeaucoup      d'espaces    superflus  \n    et    des   lignes   vides"

Après:
"Ceci est un texte avec\n\nbeaucoup d'espaces superflus\net des lignes vides"
Ceci est un texte avec

beaucoup d'espaces superflus
et des lignes vides

TEST 2: Nettoyage HTML vers texte
HTML original:

<html>
    <head>
        <title>Test Page</title>
        <script>alert('test');</script>
        <style>.test { color: red; }</style>
    </head>
    <body>
        <h1>Titre Principal</h1>
       ...

Texte nettoyé:
Test Page

Titre Principal
Ceci est un paragraphe avec des espaces.

Texte dans un span
Un lien

Entités HTML

TEST 3: Extraction de domaine
URL: https://www.esiee.fr/
→ Domaine: www.esiee.fr

URL: http://example.com/path/to/page?query=value&other=123
→ Domaine: example.com

URL: https://subdomain.example.co.uk/page
→ Domaine: subdomain.example.co.uk

URL: www.google.com
→ Domaine: google.com

URL: https:/

# Exploitation du HTML  

Ici, il faut récupérer le code HTML d'un site web à partir d'une requête. Lorsque vous avez récupéré le texte d'un site il faut le parser. Pour cela, on utilise BeautifulSoup qui permet de transformer la structure HTML en objet Python. Cela permet de récupérer efficacement les données qui nous intéresse.

Pour les webmasters, le blocage le plus souvent mis en place et un blocage sur le User-Agent. Le User-Agent est un paramètre intégré dans la requête HTTP réalisé par le Navigateur pour envoyer au front des informations basiques :

- la version du Navigateur,
- la version de l'OS
- Le type de gestionnaire graphique (Gecko)
- le type de device utilisé

Exemple de User Agent :  

`Mozilla/5.0 (Windows NT 6.1; Win64; x64; rv:47.0) Gecko/20100101 Firefox/47.0`

Commençons à utiliser `BeautifulSoup`, il est normalement déjà installé, le cas échéant executez les lignes suivantes : 

In [12]:
import requests
from bs4 import BeautifulSoup

Pour transformer une requête (requests) en objet BeautifulSoup :

In [13]:
response = requests.get(url)
soup = BeautifulSoup(response.text)

Pour trouver tous les liens d'une page, on récupère la balise `a` qui permet de gérer les liens en HTML :

In [14]:
soup.find_all("a")[0:10]

[<a class="px-2 py-4 color-bg-accent-emphasis color-fg-on-emphasis show-on-focus js-skip-to-content" data-skip-target-assigned="false" href="#start-of-content">Skip to content</a>,
 <a aria-label="Homepage" class="mr-lg-3 color-fg-inherit flex-order-2 js-prevent-focus-on-mobile-nav" data-analytics-event='{"category":"Marketing nav","action":"click to go to homepage","label":"ref_page:Marketing;ref_cta:Logomark;ref_loc:Header"}' href="/">
 <svg aria-hidden="true" class="octicon octicon-mark-github" data-view-component="true" height="32" version="1.1" viewbox="0 0 24 24" width="32">
 <path d="M12 1C5.923 1 1 5.923 1 12c0 4.867 3.149 8.979 7.521 10.436.55.096.756-.233.756-.522 0-.262-.013-1.128-.013-2.049-2.764.509-3.479-.674-3.699-1.292-.124-.317-.66-1.293-1.127-1.554-.385-.207-.936-.715-.014-.729.866-.014 1.485.797 1.691 1.128.99 1.663 2.571 1.196 3.204.907.096-.715.385-1.196.701-1.471-2.448-.275-5.005-1.224-5.005-5.432 0-1.196.426-2.186 1.128-2.956-.111-.275-.496-1.402.11-2.915 0 0 .92

On peut aussi préciser la classe HTML qu'on veut récupérer :

```python
soup.find_all(class_="<CLASS_NAME>")[0:10]
```

Ici par exemple: 

In [17]:
soup.find_all(class_="toggler")[0:5]

[]

Pour récupérer le text sans les balises HTML :

In [16]:
soup.text[0:1000]

'\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nPage not found · GitHub · GitHub\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSkip to content\n\n\n\n\n\n\n\n\n\n\n\n\n\nNavigation Menu\n\nToggle navigation\n\n\n\n\n \n\n\n\n\n\n\n\n\n\n\n\n\n\n            Sign in\n          \n\n\n \n\n\nAppearance settings\n\n\n\n\n\n\n\n\n\n\n\n        Platform\n        \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n          GitHub Copilot\n\n        \n\n        Write better code with AI\n      \n\n\n\n\n\n\n\n\n          GitHub Spark\n\n            \n              New\n            \n\n\n        Build and deploy intelligent apps\n      \n\n\n\n\n\n\n\n\n          GitHub Models\n\n            \n              New\n            \n\n\n        Manage and compare prompts\n      \n\n\n\n\n\n\n\n\n          GitHub Advanced Security\n\n        \n\n        Find and fix vulnerab

## Exercice
### Exercice 3

Améliorer la classe développé précédemment.

- Ajouter une méthode pour récupérer l'objet soup d'un url
- Récupérer une liste de User Agent et effectuer une rotation aléatoire sur celui à utiliser
- Utiliser cette classe pour parser une page HTML et récupérer : le titre, tous les H1 (si ils existent), les liens vers les images, les liens sortants vers d'autres sites, et le texte principal.

In [26]:
## Exercice 3 : Classe HTTPClient améliorée avec scraping

import requests
import time
import random
import re
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin

class HTTPClientAdvanced:
    """
    Classe avancée pour effectuer des requêtes HTTP et du web scraping
    avec rotation de User-Agent et méthodes d'extraction de données
    """
    
    def __init__(self):
        # Liste de User-Agents pour la rotation aléatoire
        self.user_agents = [
            # Chrome sur Windows
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            # Chrome sur Mac
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            # Firefox sur Windows
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:122.0) Gecko/20100101 Firefox/122.0',
            # Firefox sur Mac
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:122.0) Gecko/20100101 Firefox/122.0',
            # Safari sur Mac
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Safari/605.1.15',
            # Edge sur Windows
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36 Edg/120.0.0.0',
        ]
    
    def _get_random_user_agent(self):
        """Retourne un User-Agent aléatoire de la liste"""
        return random.choice(self.user_agents)
    
    def get(self, url, timeout=10, max_retries=3, retry_count=0):
        """
        Effectue une requête GET avec retry récursif et User-Agent aléatoire
        
        Args:
            url: URL à requêter
            timeout: Temps maximum d'attente (par défaut 10s)
            max_retries: Nombre maximum de tentatives
            retry_count: Compteur de tentatives (utilisé par la récursion)
        
        Returns:
            Response object ou None en cas d'échec
        """
        # Sélectionner un User-Agent aléatoire
        headers = {'User-Agent': self._get_random_user_agent()}
        
        try:
            response = requests.get(url, headers=headers, timeout=timeout)
            response.raise_for_status()
            return response
            
        except requests.exceptions.RequestException as e:
            if retry_count < max_retries:
                wait_time = 2 ** retry_count
                time.sleep(wait_time)
                return self.get(url, timeout, max_retries, retry_count + 1)
            else:
                print(f"✗ Échec après {max_retries + 1} tentatives pour {url}")
                return None
    
    def get_soup(self, url, timeout=10):
        """
        Récupère l'objet BeautifulSoup d'une URL
        
        Args:
            url: URL à parser
            timeout: Timeout de la requête
        
        Returns:
            BeautifulSoup object ou None en cas d'échec
        """
        response = self.get(url, timeout=timeout)
        if response:
            return BeautifulSoup(response.text, 'html.parser')
        return None
    
    def parse_page(self, url, timeout=10):
        """
        Parse une page HTML et extrait les données principales
        
        Args:
            url: URL de la page à parser
            timeout: Timeout de la requête
        
        Returns:
            Dictionnaire contenant :
            - titre: Le titre de la page (<title>)
            - h1: Liste de tous les H1
            - images: Liste des URLs des images
            - liens_externes: Liste des liens sortants vers d'autres sites
            - texte: Le texte principal nettoyé
        """
        soup = self.get_soup(url, timeout)
        if not soup:
            return None
        
        # Extraire le domaine de base pour identifier les liens externes
        parsed_url = urlparse(url)
        base_domain = parsed_url.netloc
        
        # 1. Récupérer le titre
        titre = soup.find('title')
        titre_text = titre.get_text().strip() if titre else "Pas de titre"
        
        # 2. Récupérer tous les H1
        h1_list = [h1.get_text().strip() for h1 in soup.find_all('h1')]
        
        # 3. Récupérer les liens vers les images
        images = []
        for img in soup.find_all('img', src=True):
            img_url = img['src']
            # Convertir les URLs relatives en absolues
            img_url_absolute = urljoin(url, img_url)
            images.append(img_url_absolute)
        
        # 4. Récupérer les liens sortants (vers d'autres domaines)
        liens_externes = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            # Convertir en URL absolue
            href_absolute = urljoin(url, href)
            # Vérifier si c'est un lien externe
            parsed_href = urlparse(href_absolute)
            if parsed_href.netloc and parsed_href.netloc != base_domain:
                liens_externes.append(href_absolute)
        
        # Dédupliquer les liens externes
        liens_externes = list(set(liens_externes))
        
        # 5. Extraire le texte principal
        # Supprimer les scripts, styles, etc.
        for script in soup(['script', 'style', 'meta', 'link', 'noscript']):
            script.decompose()
        
        texte = soup.get_text()
        # Nettoyer le texte
        texte = re.sub(r' +', ' ', texte)
        texte = re.sub(r'\n\s*\n+', '\n\n', texte)
        lines = [line.strip() for line in texte.split('\n')]
        texte = '\n'.join(line for line in lines if line).strip()
        
        return {
            'titre': titre_text,
            'h1': h1_list,
            'images': images,
            'liens_externes': liens_externes,
            'texte': texte[:500] + '...' if len(texte) > 500 else texte  # Limiter le texte pour l'affichage
        }


# ===== TESTS DE LA CLASSE AVANCÉE =====

print("=" * 80)
print("TEST : Scraping complet de la page ESIEE")
print("=" * 80)

scraper = HTTPClientAdvanced()
data = scraper.parse_page("https://www.esiee.fr/", timeout=10)

if data:
    print(f"\n📄 TITRE DE LA PAGE:")
    print(f"   {data['titre']}")
    
    print(f"\n📌 H1 TROUVÉS ({len(data['h1'])}):")
    for i, h1 in enumerate(data['h1'][:5], 1):  # Afficher les 5 premiers
        print(f"   {i}. {h1}")
    if len(data['h1']) > 5:
        print(f"   ... et {len(data['h1']) - 5} autres")
    
    print(f"\n🖼️  IMAGES TROUVÉES ({len(data['images'])}):")
    for i, img in enumerate(data['images'][:5], 1):  # Afficher les 5 premières
        print(f"   {i}. {img[:80]}...")
    if len(data['images']) > 5:
        print(f"   ... et {len(data['images']) - 5} autres")
    
    print(f"\n🔗 LIENS EXTERNES ({len(data['liens_externes'])}):")
    for i, link in enumerate(data['liens_externes'][:5], 1):  # Afficher les 5 premiers
        print(f"   {i}. {link}")
    if len(data['liens_externes']) > 5:
        print(f"   ... et {len(data['liens_externes']) - 5} autres")
    
    print(f"\n📝 EXTRAIT DU TEXTE:")
    print(f"   {data['texte'][:300]}...")
else:
    print("✗ Échec du scraping")

print("\n" + "=" * 80)
print("TEST : Rotation des User-Agents")
print("=" * 80)
print("Vérification que chaque requête utilise un User-Agent différent...")
print("(via httpbin.org/user-agent)")

for i in range(3):
    response = scraper.get("https://httpbin.org/user-agent", timeout=10)
    if response:
        ua = response.json()['user-agent']
        print(f"   Requête {i+1}: {ua[:60]}...")
    time.sleep(0.5)  # Pause entre les requêtes

TEST : Scraping complet de la page ESIEE

📄 TITRE DE LA PAGE:
   ESIEE Paris, l'école de l'innovation technologique | ESIEE Paris

📌 H1 TROUVÉS (1):
   1. 

🖼️  IMAGES TROUVÉES (86):
   1. https://www.esiee.fr/typo3conf/ext/esiee_sitepackage/Resources/Public/imgs/svg/l...
   2. data:image/gif;base64,R0lGODlhAQABAIAAAP///wAAACH5BAEAAAAALAAAAAABAAEAAAICRAEAOw...
   3. https://www.esiee.fr/fileadmin/user_upload/Fichiers/image-home/ESIEE-Home-Main-P...
   4. data:image/gif;base64,R0lGODlhAQABAIAAAP///wAAACH5BAEAAAAALAAAAAABAAEAAAICRAEAOw...
   5. https://www.esiee.fr/fileadmin/_processed_/0/b/csm_photos-salons-1344x840_472aff...
   ... et 81 autres

🔗 LIENS EXTERNES (12):
   1. https://www.instagram.com/esieeparis/
   2. https://www.cti-commission.fr/
   3. https://www.facebook.com/esieeparis
   4. https://www.cge.asso.fr/
   5. https://bsky.app/profile/esiee.fr
   ... et 7 autres

📝 EXTRAIT DU TEXTE:
   ESIEE Paris, l'école de l'innovation technologique | ESIEE Paris
Aller au contenu
Alle

# Exploitation des appels d'API



Losque le front du site récupère des données sur une API gérée par le back, un appel d'API est réalisé. Cet appel est recensé dans les appels réseaux. Il est alors possible de re-jouer cet appel pour récupérer à nouveau les données. Il est très facile de récupérer ces appels dans l'onglet Network de la console développeur de Chrome ou FireFox. La console vous permet de copier le code CURL de la requête et vous pouvez ensuite la transformer en code Python depuis le site https://curl.trillworks.com/.

Souvent les APIs sont bloquées avec certains paramètres. L'API vérifie que dans les headers de la requête HTTP ces paramètres sont présents :
* un token généré à la volée avec des protocoles OAuth2 (ou moins développés).
* un referer provenant du site web (la source de la requête), très facile à falsifier.



## Exercice 
### Exercice 4

- Utiliser les informations développées plus haut pour récupérer les premiers résultats d'une recherche d'une requête
sur Google. 

Tips : 

- Ouvrir les outils de développements de Chrome ou Firefox
- Onglet Network
- Fouiller dans les requêtes pour voir à quoi ressemble un appel API Google
- Utilisez beautiful soup pour convertir le contenu de la request en objet et accéder aux balises

In [ ]:
import requests
from bs4 import BeautifulSoup
response = requests.get("https://www.esiee.fr/")
soup = BeautifulSoup(response.text, 'html.parser')
open("esiee_prettify.html", "w", encoding="utf-8")




<_io.TextIOWrapper name='esiee_prettify.html' mode='w' encoding='utf-8'>

# Exercice Final  

Exercice Final
Utilisez tout ce que vous avez appris pour récupérer des articles de News avec une catégorie. Il est souvent intéressant de partir des flux RSS pour commencer :

Les données doivent comprendre :
- Le texte important propre
- L'url
- Le domaine
- la catégorie
- Le titre de l'article
- Le titre de la page
- (Facultatif) : les images

Tips : 

- Taper le nom de votre média favoris + RSS (par exemple : https://www.lemonde.fr/rss/)
- Aller dans le DOM de la page 
- Trouver les catégories et les liens vers les articles